# Chapter 4 — Exercise Solutions## Policies, Values & Bellman Equations[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 4.1: Full Policy Iteration

In [ ]:
import numpy as npimport matplotlib.pyplot as pltsize = 5goal = (4, 4)gamma = 0.9actions_delta = [(-1,0),(1,0),(0,-1),(0,1)]actions_sym   = ['↑','↓','←','→']def bellman_eval(policy, V, gamma=0.9, n_iter=200):    """Evaluate a fixed policy via Bellman iteration."""    for _ in range(n_iter):        V_new = np.copy(V)        for r in range(size):            for c in range(size):                if (r,c) == goal:                    V_new[r][c] = 10                    continue                a_idx = policy[r][c]                dr, dc = actions_delta[a_idx]                nr, nc = r+dr, c+dc                if 0<=nr<size and 0<=nc<size:                    reward = 10 if (nr,nc)==goal else -1                    V_new[r][c] = reward + gamma * V[nr][nc]                else:                    V_new[r][c] = -1 + gamma * V[r][c]        V = V_new    return Vdef greedy_policy(V):    """Extract greedy policy from value function."""    policy = np.zeros((size, size), dtype=int)    for r in range(size):        for c in range(size):            if (r,c) == goal: continue            best_val, best_a = -999, 0            for a_idx, (dr,dc) in enumerate(actions_delta):                nr, nc = r+dr, c+dc                if 0<=nr<size and 0<=nc<size:                    val = (10 if (nr,nc)==goal else -1) + gamma*V[nr][nc]                else:                    val = -1 + gamma*V[r][c]                if val > best_val:                    best_val, best_a = val, a_idx            policy[r][c] = best_a    return policy# Policy Iterationpolicy = np.random.randint(0, 4, (size, size))  # random startV = np.zeros((size, size))n_iterations = 0for iteration in range(50):    V = bellman_eval(policy, V)    new_policy = greedy_policy(V)    n_iterations += 1    if np.all(new_policy == policy):        print(f"Converged after {n_iterations} iterations!")        break    policy = new_policyprint("\nOptimal Policy:")for r in range(size):    row = ""    for c in range(size):        if (r,c) == goal: row += " ★ "        else: row += f" {actions_sym[policy[r][c]]} "    print(row)# YOUR TURN: What happens if you start with a fully greedy policy (always →)?# Does it converge in more or fewer iterations than a random policy?

### Solution 4.2: Value Functions for Different γ

In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef compute_V_random_policy(gamma, n_iter=300):    V = np.zeros((size, size))    for _ in range(n_iter):        V_new = np.copy(V)        for r in range(size):            for c in range(size):                if (r,c) == goal:                    V_new[r][c] = 10; continue                vs = []                for dr,dc in actions_delta:                    nr,nc = r+dr, c+dc                    if 0<=nr<size and 0<=nc<size:                        vs.append((10 if (nr,nc)==goal else -1) + gamma*V[nr][nc])                    else:                        vs.append(-1 + gamma*V[r][c])                V_new[r][c] = np.mean(vs)        V = V_new    return Vgammas = [0.5, 0.7, 0.9, 0.99]fig, axes = plt.subplots(1, 4, figsize=(18, 4))for ax, g in zip(axes, gammas):    V = compute_V_random_policy(g)    im = ax.imshow(V, cmap='YlOrRd', vmin=-10)    for r in range(size):        for c in range(size):            ax.text(c, r, f'{V[r][c]:.1f}', ha='center', va='center', fontsize=8)    ax.set_title(f'γ = {g}')    plt.colorbar(im, ax=ax)plt.suptitle('Value Functions for Random Policy at Different γ', fontsize=13)plt.tight_layout(); plt.show()print("Observation: higher γ makes distant cells MORE valuable (non-zero V)")print("Low γ: only cells adjacent to goal have significant value")print("High γ: cells far from goal also have meaningful value — agent plans farther ahead")# YOUR TURN: Compute the ratio V(0,0)/V(3,3) for each gamma# How does planning horizon scale with gamma?

### Solution 4.3: Q-Function Verification

In [ ]:
import numpy as np# Use V from gamma=0.9 random policyV = compute_V_random_policy(0.9)# Compute Q(s,a) = R(s,a) + gamma * V(s')Q = np.zeros((size, size, 4))  # [row, col, action]for r in range(size):    for c in range(size):        for a_idx, (dr,dc) in enumerate(actions_delta):            nr,nc = r+dr, c+dc            if 0<=nr<size and 0<=nc<size:                Q[r,c,a_idx] = (10 if (nr,nc)==goal else -1) + 0.9*V[nr][nc]            else:                Q[r,c,a_idx] = -1 + 0.9*V[r][c]# Verify: V(s) = mean_a Q(s,a) for random (uniform) policyprint("Verifying V(s) = Σ_a π(a|s) Q(s,a) for uniform policy:")max_error = 0for r in range(size):    for c in range(size):        if (r,c)==goal: continue        V_from_Q = np.mean(Q[r,c,:])  # uniform policy: equal weight        error = abs(V_from_Q - V[r][c])        max_error = max(max_error, error)print(f"Max absolute error: {max_error:.6f}  ← should be ~0")# Verify: greedy policy picks argmax_a Q(s,a)print("\nGreedy actions from Q at a few states:")for (r,c) in [(0,0),(0,4),(2,2),(3,3)]:    best_a = np.argmax(Q[r,c,:])    print(f"  State ({r},{c}): best action = {actions_sym[best_a]}, Q values = {Q[r,c,:].round(2)}")# YOUR TURN: Verify that policy_iteration using Q instead of V gives same result